# Huy Dung · ADY201m · Lưu trữ và khai thác dữ liệu SQL
Phiên bản HD-2026. Phương án SQLite thay IBM Cloud được sử dụng theo thông tin giảng viên đã chấp nhận.
Dữ liệu gốc được giữ nguyên tuổi theo ngày. Cơ sở dữ liệu lưu bền vững và có khóa chính ID.
13 truy vấn bắt buộc giữ cùng ngữ nghĩa với đề; phần bổ sung tổng hợp theo nhóm tuổi và mã giới tính.
Các nhóm giới hạn 10 dòng có thứ tự xác định; tỷ lệ của các nhóm này chỉ mô tả hồ sơ được truy vấn.

In [1]:
from pathlib import Path
import json, sqlite3
import pandas as pd
from IPython.display import display
from sqlite_storage import open_cardio_database
ROOT=Path.cwd()
raw=pd.read_csv(ROOT/'data/cardio_train_raw.csv',sep=';')
database_path=ROOT/'data/cardio_train.sqlite'
connection,created=open_cardio_database(raw,database_path,ROOT/'sql/01_schema.sql')
print('Tệp SQLite:',database_path.name,'; tạo mới:',created)
print('Số dòng:',connection.execute('SELECT COUNT(*) FROM CARDIO_TRAIN').fetchone()[0])
query_log=[]

Tệp SQLite: cardio_train.sqlite ; tạo mới: True
Số dòng: 70000


## Q01. Tổng số hồ sơ

```sql
SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN;
```

In [2]:
result=pd.read_sql_query('SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN',connection)
display(result)
query_log.append({'query':'Q01','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q01.csv',index=False)

,TOTAL_ROWS
0,70000


## Q02. Số người mắc bệnh

```sql
SELECT COUNT(*) AS DISEASE_CASES FROM CARDIO_TRAIN WHERE CARDIO=1;
```

In [3]:
result=pd.read_sql_query('SELECT COUNT(*) AS DISEASE_CASES FROM CARDIO_TRAIN WHERE CARDIO=1',connection)
display(result)
query_log.append({'query':'Q02','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q02.csv',index=False)

,DISEASE_CASES
0,34979


## Q03. Số người theo giới tính

```sql
SELECT GENDER, COUNT(*) AS PEOPLE FROM CARDIO_TRAIN GROUP BY GENDER ORDER BY GENDER;
```

In [4]:
result=pd.read_sql_query('SELECT GENDER, COUNT(*) AS PEOPLE FROM CARDIO_TRAIN GROUP BY GENDER ORDER BY GENDER',connection)
display(result)
query_log.append({'query':'Q03','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q03.csv',index=False)

,GENDER,PEOPLE
0,1,45530
1,2,24470


## Q04. 10 người có AP_HI cao nhất

```sql
SELECT ID, AP_HI, CARDIO FROM CARDIO_TRAIN ORDER BY AP_HI DESC, ID ASC LIMIT 10;
```

In [5]:
result=pd.read_sql_query('SELECT ID, AP_HI, CARDIO FROM CARDIO_TRAIN ORDER BY AP_HI DESC, ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q04','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q04.csv',index=False)

,ID,AP_HI,CARDIO
0,58374,16020,1
1,36339,14020,1
2,36414,14020,1
3,66998,14020,1
4,67502,14020,0
5,79116,13010,1
6,79679,13010,0
7,11089,11500,1
8,73356,11020,1
9,99089,2000,1


## Q05. 10 người có cân nặng trên trung bình

```sql
SELECT ID, WEIGHT, CARDIO FROM CARDIO_TRAIN WHERE WEIGHT>(SELECT AVG(WEIGHT) FROM CARDIO_TRAIN) ORDER BY WEIGHT DESC, ID ASC LIMIT 10;
```

In [6]:
result=pd.read_sql_query('SELECT ID, WEIGHT, CARDIO FROM CARDIO_TRAIN WHERE WEIGHT>(SELECT AVG(WEIGHT) FROM CARDIO_TRAIN) ORDER BY WEIGHT DESC, ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q05','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q05.csv',index=False)

,ID,WEIGHT,CARDIO
0,618,200.0,0
1,71945,200.0,1
2,87498,183.0,1
3,52564,181.0,1
4,6716,180.0,1
5,64829,180.0,1
6,86506,180.0,1
7,87612,180.0,1
8,28683,178.0,0
9,39156,178.0,1


## Q06. 10 người có đường huyết cao

```sql
SELECT ID, GLUC, CARDIO FROM CARDIO_TRAIN WHERE GLUC>1 ORDER BY ID ASC LIMIT 10;
```

In [7]:
result=pd.read_sql_query('SELECT ID, GLUC, CARDIO FROM CARDIO_TRAIN WHERE GLUC>1 ORDER BY ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q06','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q06.csv',index=False)

,ID,GLUC,CARDIO
0,8,2,0
1,12,3,1
2,27,3,0
3,32,2,1
4,40,3,0
5,68,3,1
6,71,2,0
7,92,2,1
8,103,2,0
9,117,2,1


## Q07. 10 người lớn tuổi nhất

```sql
SELECT ID, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, CARDIO FROM CARDIO_TRAIN ORDER BY AGE DESC, ID ASC LIMIT 10;
```

In [8]:
result=pd.read_sql_query('SELECT ID, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, CARDIO FROM CARDIO_TRAIN ORDER BY AGE DESC, ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q07','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q07.csv',index=False)

,ID,AGE_YEARS,CARDIO
0,81643,64,1
1,97124,64,1
2,72373,64,1
3,29892,64,1
4,52297,64,1
5,53805,64,1
6,57321,64,0
7,98749,64,1
8,25799,64,1
9,82562,64,1


## Q08. 10 người có cholesterol mức 3

```sql
SELECT ID, CHOLESTEROL, CARDIO FROM CARDIO_TRAIN WHERE CHOLESTEROL=3 ORDER BY ID ASC LIMIT 10;
```

In [9]:
result=pd.read_sql_query('SELECT ID, CHOLESTEROL, CARDIO FROM CARDIO_TRAIN WHERE CHOLESTEROL=3 ORDER BY ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q08','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q08.csv',index=False)

,ID,CHOLESTEROL,CARDIO
0,1,3,1
1,2,3,1
2,9,3,0
3,12,3,1
4,35,3,1
5,42,3,1
6,43,3,1
7,46,3,1
8,59,3,1
9,68,3,1


## Q09. 10 người hút thuốc

```sql
SELECT ID, SMOKE, CARDIO FROM CARDIO_TRAIN WHERE SMOKE=1 ORDER BY ID ASC LIMIT 10;
```

In [10]:
result=pd.read_sql_query('SELECT ID, SMOKE, CARDIO FROM CARDIO_TRAIN WHERE SMOKE=1 ORDER BY ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q09','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q09.csv',index=False)

,ID,SMOKE,CARDIO
0,23,1,0
1,29,1,0
2,52,1,1
3,73,1,1
4,74,1,1
5,90,1,1
6,138,1,0
7,140,1,1
8,142,1,1
9,166,1,1


## Q10. 10 người có vận động

```sql
SELECT ID, ACTIVE, CARDIO FROM CARDIO_TRAIN WHERE ACTIVE=1 ORDER BY ID ASC LIMIT 10;
```

In [11]:
result=pd.read_sql_query('SELECT ID, ACTIVE, CARDIO FROM CARDIO_TRAIN WHERE ACTIVE=1 ORDER BY ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q10','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q10.csv',index=False)

,ID,ACTIVE,CARDIO
0,0,1,0
1,1,1,1
2,3,1,1
3,9,1,0
4,12,1,1
5,13,1,0
6,15,1,0
7,16,1,0
8,21,1,0
9,23,1,0


## Q11. 10 người có AP_LO thấp nhất

```sql
SELECT ID, AP_LO, CARDIO FROM CARDIO_TRAIN ORDER BY AP_LO ASC, ID ASC LIMIT 10;
```

In [12]:
result=pd.read_sql_query('SELECT ID, AP_LO, CARDIO FROM CARDIO_TRAIN ORDER BY AP_LO ASC, ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q11','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q11.csv',index=False)

,ID,AP_LO,CARDIO
0,85816,-70,1
1,2845,0,0
2,19258,0,0
3,23512,0,1
4,24837,0,0
5,32749,0,1
6,34120,0,1
7,36325,0,0
8,39577,0,1
9,45400,0,0


## Q12. 10 người uống rượu

```sql
SELECT ID, ALCO, CARDIO FROM CARDIO_TRAIN WHERE ALCO=1 ORDER BY ID ASC LIMIT 10;
```

In [13]:
result=pd.read_sql_query('SELECT ID, ALCO, CARDIO FROM CARDIO_TRAIN WHERE ALCO=1 ORDER BY ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q12','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q12.csv',index=False)

,ID,ALCO,CARDIO
0,23,1,0
1,81,1,0
2,134,1,1
3,136,1,0
4,142,1,1
5,180,1,1
6,199,1,0
7,213,1,1
8,240,1,0
9,318,1,1


## Q13. Xem dữ liệu và đổi tuổi sang năm

```sql
SELECT ID, AGE AS AGE_DAYS, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, GENDER, CARDIO FROM CARDIO_TRAIN ORDER BY ID ASC LIMIT 10;
```

In [14]:
result=pd.read_sql_query('SELECT ID, AGE AS AGE_DAYS, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, GENDER, CARDIO FROM CARDIO_TRAIN ORDER BY ID ASC LIMIT 10',connection)
display(result)
query_log.append({'query':'Q13','result_rows':len(result),'disease_rate':float(result.CARDIO.mean()) if 'CARDIO' in result else None})
result.to_csv(ROOT/'results/sql_Q13.csv',index=False)

,ID,AGE_DAYS,AGE_YEARS,GENDER,CARDIO
0,0,18393,50,2,0
1,1,20228,55,1,1
2,2,18857,51,1,1
3,3,17623,48,2,1
4,4,17474,47,1,0
5,8,21914,60,1,0
6,9,22113,60,1,0
7,12,22584,61,2,1
8,13,17668,48,1,0
9,14,19834,54,1,0


## Tổng hợp bổ sung và kiểm tra lưu trữ
AGE/365 trong SQLite dùng phép chia nguyên cho cột AGE kiểu INTEGER. Phần tổng hợp không lọc 10 hồ sơ.
Sau xuất CSV, kết nối được đóng và tệp được mở lại để kiểm tra tính toàn vẹn và tổng số bản ghi.

In [15]:
supplement=pd.read_sql_query((ROOT/'sql/03_age_gender_summary.sql').read_text(encoding='utf-8'),connection)
display(supplement); supplement.to_csv(ROOT/'results/sql_age_gender.csv',index=False)
exported=pd.read_sql_query('SELECT * FROM CARDIO_TRAIN ORDER BY ID',connection)
exported.to_csv(ROOT/'data/CARDIO_TRAIN_export.csv',index=False)
pd.DataFrame(query_log).to_csv(ROOT/'results/query_summary.csv',index=False)
connection.close()
with sqlite3.connect(database_path) as reopened:
    count,cases=reopened.execute('SELECT COUNT(*),SUM(CARDIO) FROM CARDIO_TRAIN').fetchone()
    integrity=reopened.execute('PRAGMA integrity_check').fetchone()[0]
assert (count,cases,integrity)==(70000,34979,'ok')
status={'engine':'SQLite','author':'Huy Dung','variant':'HD-2026','persistent':True,'rows':count,
        'disease_cases':cases,'required_queries':13,'supplementary_queries':1,
        'reopen_verified':True,'integrity_check':integrity,'cloud_executed':False,'github_uploaded':False}
(ROOT/'results/db_execution_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
display(status)

,AGE_GROUP,GENDER,RECORDS,CASES,CASE_PERCENT
0,40-49,1,12196,4434,36.36
1,40-49,2,7429,3014,40.57
2,50-59,1,23848,12211,51.20
3,50-59,2,11693,6144,52.54
4,60+,1,8377,5716,68.23
5,60+,2,4673,3028,64.80
6,<40,1,1109,255,22.99
7,<40,2,675,177,26.22


{'engine': 'SQLite',
 'author': 'Huy Dung',
 'variant': 'HD-2026',
 'persistent': True,
 'rows': 70000,
 'disease_cases': 34979,
 'required_queries': 13,
 'supplementary_queries': 1,
 'reopen_verified': True,
 'integrity_check': 'ok',
 'cloud_executed': False,
 'github_uploaded': False}